# 27 — Cluster State Evolution

Consolidates legacy experiments 92, 93, and 94.

1. **Cluster state preparation** — prepare a cluster state in the bosonic mode (Exp 92)
2. **Cluster state evolution** — evolve the cluster state under the target Hamiltonian (Exp 93)
3. **Cluster state verification** — verify cluster state via Wigner / correlation measurements (Exp 94)

## 1. Shared Session Bootstrap

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from qualang_tools.units import unit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "qubox").exists():
    REPO_ROOT = Path(r"E:\qubox")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qubox.notebook import (
    StorageWignerTomography,
    load_stage_checkpoint,
    open_notebook_stage,
    save_stage_checkpoint,
)

REGISTRY_BASE = Path(r"E:\qubox")
SAMPLE_ID = "post_cavity_sample_A"
COOLDOWN_ID = "cd_2025_02_22"
QOP_IP = "10.157.36.68"
CLUSTER_NAME = "Cluster_2"

stage = open_notebook_stage(
    stage_name="27_cluster_state_evolution",
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    qop_ip=QOP_IP,
    cluster_name=CLUSTER_NAME,
    force_reopen=True,
    close_existing=True,
)
session = stage.session
attr = stage.attr
SESSION_BOOTSTRAP_PATH = stage.bootstrap_path
u = unit(coerce_to_integer=True)

prev_checkpoint = load_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="26_sequential_simulation",
)

print(f"Repo root on sys.path: {REPO_ROOT}")
print(f"Shared session bootstrap: {SESSION_BOOTSTRAP_PATH}")
print(f"Stage checkpoint path: {stage.checkpoint_path}")
print(f"QM endpoint: {QOP_IP} ({CLUSTER_NAME})")
if prev_checkpoint is not None:
    print(f"Prior stage 26 status: {prev_checkpoint['status']}")

## 2. Cluster State Defaults

In [ ]:
CLUSTER_N_AVG = 10000
CLUSTER_EVOLUTION_TIMES_US = [0, 5, 10, 20, 50]

# Wigner verification
WIGNER_ALPHA_MAX = 3.0
WIGNER_ALPHA_STEP = 0.1
WIGNER_N_AVG = 10000

print("Cluster state settings:")
print(f"  n_avg: {CLUSTER_N_AVG}")
print(f"  evolution times (μs): {CLUSTER_EVOLUTION_TIMES_US}")
print(f"  Wigner: α_max={WIGNER_ALPHA_MAX}, dα={WIGNER_ALPHA_STEP}")

## 3. Cluster State Preparation — Exp 92

Prepare a cluster state in the bosonic mode.

In [ ]:
cluster_prep_result = session.cluster_state_preparation(
    n_avg=CLUSTER_N_AVG,
)

print("Cluster state preparation complete.")

## 4. Cluster State Evolution — Exp 93

Evolve the cluster state under the target Hamiltonian at various times.

In [ ]:
cluster_evol_results = {}

for t_us in CLUSTER_EVOLUTION_TIMES_US:
    result = session.cluster_state_evolution(
        t_evolve_us=t_us,
        n_avg=CLUSTER_N_AVG,
    )
    cluster_evol_results[t_us] = result
    print(f"  t={t_us} μs: done")

print("Cluster state evolution sweep complete.")

## 5. Cluster State Verification — Exp 94

Verify the cluster state via Wigner tomography at selected evolution time.

In [ ]:
wigner_cluster = StorageWignerTomography(session)
wigner_cluster_result = wigner_cluster.run(
    alpha_max=WIGNER_ALPHA_MAX,
    alpha_step=WIGNER_ALPHA_STEP,
    n_avg=WIGNER_N_AVG,
    state_prep="cluster",
)
wigner_cluster_analysis = wigner_cluster.analyze(
    wigner_cluster_result, update_calibration=False
)
wigner_cluster.plot(wigner_cluster_analysis)

print("Cluster state Wigner verification complete.")

## 6. Save Checkpoint

In [ ]:
stage_checkpoint_path = save_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="27_cluster_state_evolution",
    status="characterized",
    summary="Cluster state preparation, evolution at multiple times, and Wigner verification.",
    consumed_inputs={
        "cluster_n_avg": CLUSTER_N_AVG,
        "cluster_evolution_times_us": CLUSTER_EVOLUTION_TIMES_US,
        "wigner_alpha_max": WIGNER_ALPHA_MAX,
    },
    persisted_outputs={},
    advisory_outputs={},
    next_stage=None,
    notes=[
        "Terminal notebook in the calibration pipeline.",
        "All experiments are characterization-only.",
        "Cluster state fidelity depends on all prior calibrations.",
    ],
    metrics={},
)

print(f"Stage checkpoint saved to: {stage_checkpoint_path}")